# Classification result figures

Set `DEBUG = True` in the setup cell to read/write under `Temp`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Daprosero/STF-KernelSHAP.git"
REPO_NAME = "STF-KernelSHAP"
PACKAGE_PATH = Path("src") / "stf_kernelshap"


def resolve_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / PACKAGE_PATH).exists():
            return candidate

    working_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else cwd
    clone_dir = working_root / REPO_NAME
    if not clone_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
    return clone_dir.resolve()


def install_project_requirements(project_root):
    requirements_path = project_root / "requirements.txt"
    if not requirements_path.exists():
        raise FileNotFoundError(f"No existe requirements.txt en {project_root}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)],
        check=True,
    )


PROJECT_ROOT = resolve_project_root()
os.chdir(PROJECT_ROOT)

RUNNING_IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
RUNNING_IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").exists()
INSTALL_REQUIREMENTS = RUNNING_IN_COLAB or RUNNING_IN_KAGGLE

if INSTALL_REQUIREMENTS:
    install_project_requirements(PROJECT_ROOT)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INSTALL_REQUIREMENTS:", INSTALL_REQUIREMENTS)


In [ ]:
from stf_kernelshap.notebook_setup import setup_notebook_environment

DEBUG = True
paths = setup_notebook_environment(debug=DEBUG)

DATA_DIR = paths.data_dir
MODELS_DIR = paths.models_dir
OUTPUT_MODELS_DIR = paths.output_models_dir
RESULTS_DIR = paths.results_dir
FIGURES_DIR = paths.figures_dir


In [ ]:
from stf_kernelshap.reporting.classification_results import (
    plot_mi_window_figure,
    plot_tdah_bar_figure,
    summarize_mi_global,
    summarize_mi_subject_level,
    summarize_tdah_global,
)


In [ ]:
mi_csv_path = RESULTS_DIR / "MI_results.csv"
tdah_csv_path = RESULTS_DIR / "TDAH_results.csv"

mi_global_summary = summarize_mi_global(mi_csv_path)
mi_subject_summary = summarize_mi_subject_level(mi_csv_path)
tdah_summary = summarize_tdah_global(tdah_csv_path)


In [ ]:
plot_mi_window_figure(
    window_name="2.5-5",
    mi_global_summary=mi_global_summary,
    mi_subject_summary=mi_subject_summary,
    save_path=str(FIGURES_DIR / "Cl_MI_2_5_5.pdf"),
)

plot_mi_window_figure(
    window_name="0-7",
    mi_global_summary=mi_global_summary,
    mi_subject_summary=mi_subject_summary,
    save_path=str(FIGURES_DIR / "Cl_MI_0_7.pdf"),
)

plot_tdah_bar_figure(
    tdah_summary=tdah_summary,
    save_path=str(FIGURES_DIR / "Cl_TDAH.pdf"),
)
